# Etapa 4:  Motor Inteligente de Decisión para Reabastecimiento

**Responsable:** Dalma Leguizamón — Data Engineer & Machine Learning Analyst

**Objetivo:** Diseñar e implementar un motor de inteligencia aplicado a inventario que analice el stock disponible, la velocidad de consumo, el lead time de proveedores y el impacto económico de cada producto para determinar **cuándo comprar, qué tan urgente es la compra y cuál sería la pérdida financiera estimada en caso de no reponer**.

Este motor no busca predecir ventas futuras mediante entrenamiento de modelos, sino generar decisiones operativas basadas en métricas dinámicas del negocio (cobertura, riesgo de quiebre, ABC y revenue implícito), priorizando automáticamente los productos con mayor impacto económico y facilitando la toma de decisiones estratégicas de abastecimiento.



# Imports + settings

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Cargar archivos

In [10]:
# Carda del DataSet
purchases = pd.read_csv("PurchasesFINAL12312016.csv")
abc = pd.read_csv("productos_abc_final_con_rotacion_limpio.csv")
beg_inv = pd.read_csv("BegInvFINAL12312016.csv")
end_inv = pd.read_csv("EndInvFINAL12312016.csv")
proveedores = pd.read_csv("dim_proveedores.csv")


# Forzar fechas

In [11]:
beg_inv['startDate'] = pd.to_datetime(beg_inv['startDate'])
end_inv['endDate'] = pd.to_datetime(end_inv['endDate'])

# Se creó una lista con los nombres de las columnas a convertir
Fechas_compras_final = ["PODate", "ReceivingDate", "InvoiceDate","PayDate"]

# Se aplicó la conversión a esas columnas específicas
purchases[Fechas_compras_final] = purchases[Fechas_compras_final].apply(pd.to_datetime)

In [14]:
abc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11503 entries, 0 to 11502
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Brand                 11503 non-null  int64  
 1   Consumo               11503 non-null  float64
 2   PrecioRepresentativo  11503 non-null  float64
 3   RevenueImplicito_USD  11503 non-null  float64
 4   CategoriaABC          11503 non-null  object 
 5   Rotacion              9454 non-null   float64
 6   CategoriaRotacion     11503 non-null  object 
dtypes: float64(4), int64(1), object(2)
memory usage: 629.2+ KB


# Normalizar llaves (Brand y VendorNumber)

In [ ]:
# VendorNumber: aseguramos numérico

purchases["VendorNumber"] = pd.to_numeric(purchases["VendorNumber"], errors="coerce").astype("Int64")
proveedores["VendorNumber"] = pd.to_numeric(proveedores["VendorNumber"], errors="coerce").astype("Int64")

# Brand: aseguramos numérico (en tu ABC ya es int64, igual lo normalizamos)
purchases["Brand"] = pd.to_numeric(purchases["Brand"], errors="coerce").astype("Int64")
abc["Brand"] = pd.to_numeric(abc["Brand"], errors="coerce").astype("Int64")

print("OK dtypes:")
print("purchases VendorNumber:", purchases["VendorNumber"].dtype, "| Brand:", purchases["Brand"].dtype)
print("proveedores VendorNumber:", proveedores["VendorNumber"].dtype)
print("abc Brand:", abc["Brand"].dtype)

OK dtypes:
purchases VendorNumber: Int64 | Brand: Int64
proveedores VendorNumber: Int64
abc Brand: Int64


# Dataset maestro Brand–Vendor (mapa) + proveedores + ABC

In [ ]:
# Creamos DataSet maestro

# Mapa Brand-Vendor
brand_vendor_map = (
    purchases[["Brand", "VendorNumber", "VendorName"]]
    .dropna(subset=["Brand", "VendorNumber"])
    .drop_duplicates()
)

# Proveedores (lead time ya calculado / categorías ya hechas)
prov_small = (
    proveedores[["VendorNumber", "VendorName", "CategoriaEntrega", "SubcategoriaEntrega"]]
    .dropna(subset=["VendorNumber"])
    .drop_duplicates(subset=["VendorNumber"])
)

# ABC (lo que necesitamos para decisiones)
abc_small = (
    abc[["Brand", "CategoriaABC", "Consumo", "CategoriaRotacion", "PrecioRepresentativo", "RevenueImplicito_USD"]]
    .dropna(subset=["Brand"])
    .drop_duplicates(subset=["Brand"])
)

# Merge final
base_engine = (
    brand_vendor_map
    .merge(prov_small.drop(columns=["VendorName"], errors="ignore"), on="VendorNumber", how="left")
    .merge(abc_small, on="Brand", how="left")
)

# LeadTime que usa el modelo
base_engine["LeadTimePredictivo"] = pd.to_numeric(base_engine["SubcategoriaEntrega"], errors="coerce")

base_engine.head()

,Brand,VendorNumber,VendorName,CategoriaEntrega,SubcategoriaEntrega,CategoriaABC,Consumo,CategoriaRotacion,PrecioRepresentativo,RevenueImplicito_USD,LeadTimePredictivo
0,8412,105,ALTAMAR BRANDS LLC,Moderado,8,B,307.0,Baja,35.71,10962.97,8
1,5255,4466,AMERICAN VINTAGE BEVERAGE,Moderado,7,A,6096.0,Media,9.35,56997.60,7
2,5215,4466,AMERICAN VINTAGE BEVERAGE,Moderado,7,A,4651.0,Media,9.41,43765.91,7
3,2034,388,ATLANTIC IMPORTING COMPANY,Moderado,8,B,1587.0,Baja,21.32,33834.84,8
4,3348,480,BACARDI USA INC,Moderado,8,A,56888.0,Media,22.38,1273153.44,8


# Stock inicial y final por Brand + merge al motor

In [53]:
engine = (
    brand_vendor_map
    .merge(prov_small.drop(columns=["VendorName"], errors="ignore"), on="VendorNumber", how="left")
    .merge(abc_small, on="Brand", how="left")
)

engine["LeadTimePredictivo"] = pd.to_numeric(engine["SubcategoriaEntrega"], errors="coerce")

In [54]:
# sumamos stock por producto a través de todas las tiendas.

stock_beg = (
    beg_inv.groupby("Brand", as_index=False)["onHand"]
    .sum()
    .rename(columns={"onHand": "StockInicial"})
)

stock_end = (
    end_inv.groupby("Brand", as_index=False)["onHand"]
    .sum()
    .rename(columns={"onHand": "StockFinal"})
)

engine = (
    engine
    .merge(stock_beg, on="Brand", how="left")
    .merge(stock_end, on="Brand", how="left")
)

# Stock faltante a 0 (para poder calcular)
engine["StockInicial"] = engine["StockInicial"].fillna(0).astype(float)
engine["StockFinal"]   = engine["StockFinal"].fillna(0).astype(float)

# Calcular días del período (para consumo diario)

In [55]:
start = beg_inv["startDate"].dropna().min()
end = end_inv["endDate"].dropna().max()

days = (end - start).days
days = max(int(days), 1)

print("Periodo:", start, "->", end)
print("Días del periodo:", days)

Periodo: 2016-01-01 00:00:00 -> 2016-12-31 00:00:00
Días del periodo: 365


# Consumo diario + cobertura

In [56]:
engine["Consumo"] = pd.to_numeric(engine["Consumo"], errors="coerce")
engine["ConsumoDiario"] = engine["Consumo"] / days

engine["DiasCobertura"] = np.where(
    engine["ConsumoDiario"].fillna(0) > 0,
    engine["StockFinal"] / engine["ConsumoDiario"],
    np.nan
)

# Riesgo de quiebre + pérdida estimada 

In [57]:
# Gap vs lead time
engine["GapDias"] = engine["LeadTimePredictivo"] - engine["DiasCobertura"]
engine["DiasQuiebre"] = np.where(engine["GapDias"] > 0, engine["GapDias"], 0)

# Unidades perdidas estimadas durante quiebre
engine["UnidadesPerdidasEst"] = np.where(
    (engine["ConsumoDiario"].fillna(0) > 0) & (engine["DiasQuiebre"] > 0),
    engine["ConsumoDiario"] * engine["DiasQuiebre"],
    0
)

# Pérdida monetaria estimada
engine["PrecioRepresentativo"] = pd.to_numeric(engine["PrecioRepresentativo"], errors="coerce").fillna(0)
engine["PerdidaUSD_Est"] = engine["UnidadesPerdidasEst"] * engine["PrecioRepresentativo"]

# Recomendación mínima (solo para filtrar)
engine["Recomendacion"] = np.where(engine["DiasQuiebre"] > 0, "COMPRAR YA", "OK")
print(engine["Recomendacion"].value_counts(dropna=False))

Recomendacion
OK            8956
COMPRAR YA    1737
Name: count, dtype: int64


# RiskScore sin INF

In [58]:
# Evitar inf por división por cero
den = engine["DiasCobertura"].replace(0, np.nan)
ratio = (engine["LeadTimePredictivo"] / den).replace([np.inf, -np.inf], np.nan).fillna(0)

engine["RevenueImplicito_USD"] = pd.to_numeric(engine["RevenueImplicito_USD"], errors="coerce").fillna(0)

engine["RiskScore"] = (
    ratio
    * np.log1p(engine["Consumo"].fillna(0))
    * np.log1p(engine["RevenueImplicito_USD"])
)

#  Top productos en riesgo (para dashboard)

In [59]:
top_productos = (
    engine[engine["Recomendacion"] == "COMPRAR YA"]
    .sort_values(["PerdidaUSD_Est", "RiskScore"], ascending=[False, False])
    .loc[:, [
        "Brand","VendorNumber","VendorName",
        "CategoriaABC","CategoriaEntrega",
        "LeadTimePredictivo","StockFinal",
        "Consumo","ConsumoDiario","DiasCobertura",
        "DiasQuiebre","UnidadesPerdidasEst","PerdidaUSD_Est",
        "RiskScore"
    ]]
    .head(30)
)

top_productos

,Brand,VendorNumber,VendorName,CategoriaABC,CategoriaEntrega,LeadTimePredictivo,StockFinal,Consumo,ConsumoDiario,DiasCobertura,DiasQuiebre,UnidadesPerdidasEst,PerdidaUSD_Est,RiskScore
6148,4260,3960,DIAGEO NORTH AMERICA INC,A,Moderado,8,37.0,79042.0,216.553425,0.170859,7.829141,1695.427397,28245.820438,7440.629250
16,4227,480,BACARDI USA INC,A,Moderado,8,397.0,75147.0,205.882192,1.928287,6.071713,1250.057534,17713.315260,646.438907
956,1476,12546,JIM BEAM BRANDS COMPANY,A,Moderado,8,0.0,15044.0,41.216438,0.000000,8.000000,329.731507,5292.190685,0.000000
5,8358,480,BACARDI USA INC,A,Moderado,8,0.0,8065.0,22.095890,0.000000,8.000000,176.767123,2561.355616,0.000000
7499,2327,3960,DIAGEO NORTH AMERICA INC,A,Moderado,8,1.0,5494.0,15.052055,0.066436,7.933564,119.416438,2095.758493,11900.847150
1326,36380,4425,MARTIGNETTI COMPANIES,A,Moderado,8,6.0,14668.0,40.186301,0.149305,7.850695,315.490411,2047.532767,5892.751944
7383,405,1128,BROWN-FORMAN CORP,A,Moderado,8,0.0,3235.0,8.863014,0.000000,8.000000,70.904110,2017.221918,0.000000
3793,25588,2000,SOUTHERN WINE & SPIRITS NE,A,Moderado,8,2.0,3957.0,10.841096,0.184483,7.815517,84.728767,1716.604822,4056.148644
7500,2255,3960,DIAGEO NORTH AMERICA INC,A,Moderado,8,2.0,2379.0,6.517808,0.306852,7.693148,50.142466,1542.382247,2270.375501
1493,24631,4425,MARTIGNETTI COMPANIES,A,Moderado,8,10.0,4595.0,12.589041,0.794342,7.205658,90.712329,1521.245753,955.661607


# Impacto financiero por proveedor (Top proveedores en riesgo)

In [60]:
impacto_proveedor = (
    engine[engine["Recomendacion"] == "COMPRAR YA"]
    .groupby(["VendorNumber","VendorName"], as_index=False)
    .agg(
        PerdidaUSD_Total=("PerdidaUSD_Est","sum"),
        UnidadesPerdidas_Total=("UnidadesPerdidasEst","sum"),
        SKUs_En_Riesgo=("Brand","nunique"),
        RiskScore_Promedio=("RiskScore","mean")
    )
    .sort_values("PerdidaUSD_Total", ascending=False)
)

impacto_proveedor.head(20)

,VendorNumber,VendorName,PerdidaUSD_Total,UnidadesPerdidas_Total,SKUs_En_Riesgo,RiskScore_Promedio
33,3960,DIAGEO NORTH AMERICA INC,41123.290466,4104.339726,74,603.690765
2,480,BACARDI USA INC,24322.282000,1922.830137,28,271.456298
35,4425,MARTIGNETTI COMPANIES,22004.727260,2640.298630,191,99.537883
64,9165,ULTRA BEVERAGE COMPANY LLP,15050.320986,1473.920548,179,52.564410
79,12546,JIM BEAM BRANDS COMPANY,13539.079288,2306.041096,72,76.958852
67,9552,M S WALKER INC,11009.958795,1325.090411,141,102.045194
76,10754,PERFECTA WINES,5712.765945,682.701370,107,28.256307
7,1128,BROWN-FORMAN CORP,5595.620986,371.761644,29,306.335812
19,2000,SOUTHERN WINE & SPIRITS NE,3433.721315,333.158904,51,114.514247
11,1392,CONSTELLATION BRANDS INC,2999.478466,541.010959,43,127.160168


# Importamos los resultados 

In [62]:
#  Redondeo para presentación

top_productos["PerdidaUSD_Est"] = top_productos["PerdidaUSD_Est"].round(2)
top_productos["ConsumoDiario"]  = top_productos["ConsumoDiario"].round(2)
top_productos["DiasCobertura"]  = top_productos["DiasCobertura"].round(2)
top_productos["DiasQuiebre"]    = top_productos["DiasQuiebre"].round(2)

impacto_proveedor["PerdidaUSD_Total"] = impacto_proveedor["PerdidaUSD_Total"].round(2)

# EXPORTAR DATASETS

top_productos.to_csv("top_productos_riesgo.csv", index=False)
impacto_proveedor.to_csv("impacto_proveedor.csv", index=False)

print("✅ Exportados: top_productos_riesgo.csv y impacto_proveedor.csv")

✅ Exportados: top_productos_riesgo.csv y impacto_proveedor.csv
